# **Enron Email Data Extraction and Deduplication Pipeline**

This notebook implements an end-to-end data engineering pipeline for the Enron Email Dataset. The goal is to extract structured information from raw email files, store the extracted records in a normalized SQLite database, detect potential duplicate emails, and generate notification drafts for flagged duplicate messages.

The pipeline covers four main stages:

1. Discovering and parsing raw Enron email files
2. Storing extracted email data in a normalized SQLite database
3. Detecting duplicate emails using sender, normalized subject, and fuzzy body similarity
4. Generating dry-run notification emails and output reports

The implementation is designed to be reproducible, fault tolerant, and transparent. Parse failures are logged separately, database constraints prevent exact duplicate inserts, and output files are generated for validation and submission. This follows the assignment requirements from the provided take-home document.

# **1. Environment Setup**

This section installs the Python libraries used in the notebook.

"python-dateutil"  helps parse inconsistent email date formats, and "rapidfuzz" is used for efficient fuzzy text similarity during duplicate detection.

In [1]:
!pip install python-dateutil rapidfuzz -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 77.9 MB/s eta 0:00:00


# **2. Import Libraries and Configure Project Paths**
In this section, we import the required Python libraries and define the main project directories.

The notebook writes all generated files into one project folder so that the database, reports, logs, and draft notification emails are easy to find after execution.

In [2]:
import os
import re
import csv
import json
import time
import shutil
import tarfile
import sqlite3
import hashlib
import logging
import urllib.request

from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple
from email import policy
from email.parser import BytesParser
from email.utils import getaddresses

from dateutil import parser as date_parser
from rapidfuzz import fuzz

In [3]:
BASE_DIR = Path("/content/enron_email_pipeline")

DATA_DIR = BASE_DIR / "data"
RAW_MAILDIR = DATA_DIR / "maildir"
SUBSET_MAILDIR = DATA_DIR / "maildir_subset"

DATABASE_DIR = BASE_DIR / "database"
OUTPUT_DIR = BASE_DIR / "output"
REPLIES_DIR = OUTPUT_DIR / "replies"

DB_PATH = DATABASE_DIR / "enron_emails.db"
ERROR_LOG_PATH = OUTPUT_DIR / "error_log.txt"
DUPLICATES_REPORT_PATH = OUTPUT_DIR / "duplicates_report.csv"
SEND_LOG_PATH = OUTPUT_DIR / "send_log.csv"
EXTRACTION_STATS_PATH = OUTPUT_DIR / "extraction_stats.json"
SCHEMA_PATH = OUTPUT_DIR / "schema.sql"
SAMPLE_QUERIES_PATH = OUTPUT_DIR / "sample_queries.sql"

for folder in [DATA_DIR, DATABASE_DIR, OUTPUT_DIR, REPLIES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("enron_pipeline")

CONFIG = {
    "dataset_url": "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz",
    "selected_mailboxes": ["lay-k", "skilling-j", "kaminski-v", "mann-k", "taylor-m"],
    "minimum_email_count": 10000,
    "duplicate_similarity_threshold": 90,
}

print("Base directory:", BASE_DIR)
print("Database path:", DB_PATH)
print("Output directory:", OUTPUT_DIR)

Base directory: /content/enron_email_pipeline
Database path: /content/enron_email_pipeline/database/enron_emails.db
Output directory: /content/enron_email_pipeline/output


# **3. Download and Prepare the Enron Dataset**
The Enron dataset is distributed as a compressed archive containing raw email files organized by employee mailbox. This section downloads the dataset from CMU and extracts it into the project data directory.

For the assignment, a representative subset of at least five employee mailboxes is sufficient. The notebook selects five mailboxes and stores them separately in "maildir_subset" for reproducible processing.

In [4]:
def download_enron_dataset(force_download: bool = False) -> None:
    archive_path = DATA_DIR / "enron_mail_20150507.tar.gz"

    if RAW_MAILDIR.exists() and any(RAW_MAILDIR.iterdir()) and not force_download:
        logger.info("Raw Enron maildir already exists. Skipping download.")
        return

    if archive_path.exists() and force_download:
        archive_path.unlink()

    logger.info("Downloading Enron dataset. This may take a few minutes.")

    urllib.request.urlretrieve(CONFIG["dataset_url"], archive_path)

    logger.info("Extracting dataset archive.")

    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(DATA_DIR)

    extracted_maildir = DATA_DIR / "maildir"

    if not extracted_maildir.exists():
        raise FileNotFoundError("Expected maildir folder was not found after extraction.")

    logger.info("Dataset downloaded and extracted successfully.")

In [5]:
download_enron_dataset(force_download=False)

/tmp/ipykernel_1057/3159613156.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DATA_DIR)


# **4. Create a Representative Mailbox Subset**

The full Enron dataset contains hundreds of thousands of emails. To keep this notebook practical while still meeting the assignment requirement, this section creates a subset using five employee mailboxes.

The selected mailboxes are:

* lay-k
* skilling-j
* kaminski-v
* mann-k
* taylor-m

These mailboxes are useful because they contain a mix of executive, operational, and employee-level communication patterns.

In [6]:
def create_mailbox_subset(selected_mailboxes: List[str], overwrite: bool = False) -> None:
    if SUBSET_MAILDIR.exists() and overwrite:
        shutil.rmtree(SUBSET_MAILDIR)

    SUBSET_MAILDIR.mkdir(parents=True, exist_ok=True)

    copied_mailboxes = []

    for mailbox in selected_mailboxes:
        source = RAW_MAILDIR / mailbox
        destination = SUBSET_MAILDIR / mailbox

        if not source.exists():
            logger.warning("Selected mailbox not found: %s", mailbox)
            continue

        if destination.exists():
            logger.info("Mailbox already exists in subset: %s", mailbox)
        else:
            shutil.copytree(source, destination)
            logger.info("Copied mailbox into subset: %s", mailbox)

        copied_mailboxes.append(mailbox)

    if not copied_mailboxes:
        raise RuntimeError("No selected mailboxes were copied. Check mailbox names.")

    logger.info("Subset preparation complete.")

In [7]:
create_mailbox_subset(CONFIG["selected_mailboxes"], overwrite=False)

# **5. Discover Email Files**

This section recursively scans the selected Enron mailbox subset and collects all email files.

The Enron dataset stores emails as plain files inside nested folders such as inbox, sent mail, deleted items, and other user-created folders. The scanner ignores directories and hidden files.

In [8]:
def discover_email_files(maildir_path: Path) -> List[Path]:
    if not maildir_path.exists():
        raise FileNotFoundError(f"Maildir path does not exist: {maildir_path}")

    email_files = []

    for path in maildir_path.rglob("*"):
        if path.is_file() and not path.name.startswith("."):
            email_files.append(path)

    return sorted(email_files)

In [9]:
email_files = discover_email_files(SUBSET_MAILDIR)

print("Total email files discovered:", len(email_files))
print("First five files:")
for path in email_files[:5]:
    print(path.relative_to(SUBSET_MAILDIR))

Total email files discovered: 75797
First five files:
kaminski-v/_sent_mail/1.
kaminski-v/_sent_mail/10.
kaminski-v/_sent_mail/100.
kaminski-v/_sent_mail/1000.
kaminski-v/_sent_mail/1001.


# **6. Email Parsing Helper Functions**

Raw Enron emails follow RFC 2822 formatting, but the dataset contains real-world inconsistencies such as missing headers, malformed dates, multiline address fields, and nested forwarded messages.

The helper functions below handle the most important parsing tasks:

* Decode raw email files safely
* Normalize dates to UTC
* Extract sender and recipient email addresses
* Separate primary body text from quoted or forwarded content
* Extract optional Enron-specific headers
* Validate mandatory fields before storing records

In [10]:
def read_email_bytes(file_path: Path) -> bytes:
    with open(file_path, "rb") as file:
        return file.read()


def parse_message(raw_bytes: bytes):
    return BytesParser(policy=policy.default).parsebytes(raw_bytes)


def clean_text(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None

    value = str(value).replace("\x00", "").strip()
    return value if value else None


def normalize_date_to_utc(date_value: Optional[str]) -> Optional[str]:
    if not date_value:
        return None

    try:
        parsed_date = date_parser.parse(str(date_value), fuzzy=True)

        if parsed_date.tzinfo is None:
            parsed_date = parsed_date.replace(tzinfo=timezone.utc)

        return parsed_date.astimezone(timezone.utc).isoformat()
    except Exception:
        return None


def extract_addresses(header_value: Optional[str]) -> List[str]:
    if not header_value:
        return []

    extracted = []

    for _, address in getaddresses([str(header_value)]):
        address = address.strip().lower()

        if "@" in address and "." in address.split("@")[-1]:
            extracted.append(address)

    return sorted(set(extracted))


def normalize_subject(subject: Optional[str]) -> str:
    if not subject:
        return ""

    normalized = subject.strip()

    while True:
        updated = re.sub(
            r"^\s*(re|fw|fwd|forward)\s*:\s*",
            "",
            normalized,
            flags=re.IGNORECASE
        )

        if updated == normalized:
            break

        normalized = updated

    normalized = re.sub(r"\s+", " ", normalized).strip().lower()
    return normalized


def normalize_body_for_matching(body: Optional[str]) -> str:
    if not body:
        return ""

    body = body.lower()
    body = re.sub(r"\s+", " ", body)
    body = re.sub(r"[-_=]{3,}", " ", body)
    return body.strip()

def create_body_hash(body_normalized: str) -> str:
    return hashlib.md5(body_normalized.encode("utf-8")).hexdigest()

In [11]:
def extract_body_parts(message) -> Tuple[str, str, str]:
    body_parts = []

    if message.is_multipart():
        for part in message.walk():
            content_disposition = part.get_content_disposition()
            content_type = part.get_content_type()

            if content_disposition == "attachment":
                continue

            if content_type == "text/plain":
                try:
                    content = part.get_content()
                    if content:
                        body_parts.append(str(content))
                except Exception:
                    continue
    else:
        try:
            content = message.get_content()
            if content:
                body_parts.append(str(content))
        except Exception:
            payload = message.get_payload(decode=True)
            if payload:
                body_parts.append(payload.decode("utf-8", errors="replace"))

    full_body = "\n".join(body_parts).strip()

    forwarded_content = ""
    quoted_content = ""

    forwarded_patterns = [
        r"[-]+\s*Forwarded by.*",
        r"[-]+\s*Original Message\s*[-]+",
        r"Begin forwarded message:",
        r"Forwarded message"
    ]

    for pattern in forwarded_patterns:
        match = re.search(pattern, full_body, flags=re.IGNORECASE | re.DOTALL)

        if match:
            forwarded_content = full_body[match.start():].strip()
            full_body = full_body[:match.start()].strip()
            break

    lines = full_body.splitlines()
    primary_lines = []
    quoted_lines = []

    for line in lines:
        if line.strip().startswith(">"):
            quoted_lines.append(line)
        else:
            primary_lines.append(line)

    if quoted_lines:
        quoted_content = "\n".join(quoted_lines).strip()
        full_body = "\n".join(primary_lines).strip()

    return full_body, forwarded_content, quoted_content


def infer_has_attachment(message, body: Optional[str]) -> bool:
    for part in message.walk():
        if part.get_content_disposition() == "attachment":
            return True

        filename = part.get_filename()
        if filename:
            return True

    body = body or ""

    attachment_terms = [
        "attached",
        "attachment",
        ".doc",
        ".xls",
        ".pdf",
        ".ppt",
        ".zip"
    ]

    return any(term in body.lower() for term in attachment_terms)


def extract_headings(body: Optional[str]) -> Optional[str]:
    if not body:
        return None

    headings = []

    for line in body.splitlines():
        stripped = line.strip()

        if not stripped:
            continue

        if re.match(r"^[A-Z][A-Za-z\s]{2,40}:$", stripped):
            headings.append(stripped)

        elif re.match(r"^(from|to|subject|date|cc|bcc):", stripped, flags=re.IGNORECASE):
            headings.append(stripped)

    return "\n".join(headings) if headings else None

# **7. Parse One Email File**

This function extracts all mandatory and optional fields from one raw email file.

Mandatory fields are strictly validated. If any mandatory field is missing or cannot be parsed, the email is not inserted into the database and the failure reason is logged.

This is important because the assignment specifically requires failed mandatory-field extraction to be handled through error logging rather than silently storing incomplete records.

In [12]:
def parse_email_file(file_path: Path, root_path: Path) -> Tuple[Optional[Dict], Optional[str]]:
    try:
        raw_bytes = read_email_bytes(file_path)
        message = parse_message(raw_bytes)

        message_id = clean_text(message.get("Message-ID"))
        date = normalize_date_to_utc(message.get("Date"))
        from_addresses = extract_addresses(message.get("From"))
        to_addresses = extract_addresses(message.get("To"))
        subject = clean_text(message.get("Subject"))
        body, forwarded_content, quoted_content = extract_body_parts(message)

        if not message_id:
            return None, "Missing mandatory field: message_id"

        if not date:
            return None, "Missing or invalid mandatory field: date"

        if not from_addresses:
            return None, "Missing mandatory field: from_address"

        if not to_addresses:
            return None, "Missing mandatory field: to_addresses"

        if not subject:
            return None, "Missing mandatory field: subject"

        if not body:
            return None, "Missing mandatory field: body"

        source_file = str(file_path.relative_to(root_path))

        email_record = {
            "message_id": message_id,
            "date": date,
            "from_address": from_addresses[0],
            "to_addresses": to_addresses,
            "subject": subject,
            "subject_normalized": normalize_subject(subject),
            "body": body,
            "body_normalized": normalize_body_for_matching(body),
            "body_hash": create_body_hash(normalize_body_for_matching(body)),
            "source_file": source_file,
            "cc_addresses": extract_addresses(message.get("Cc")),
            "bcc_addresses": extract_addresses(message.get("Bcc")),
            "x_from": clean_text(message.get("X-From")),
            "x_to": clean_text(message.get("X-To")),
            "x_cc": clean_text(message.get("X-cc")),
            "x_bcc": clean_text(message.get("X-bcc")),
            "x_folder": clean_text(message.get("X-Folder")),
            "x_origin": clean_text(message.get("X-Origin")),
            "content_type": clean_text(message.get_content_type()),
            "has_attachment": infer_has_attachment(message, body),
            "forwarded_content": clean_text(forwarded_content),
            "quoted_content": clean_text(quoted_content),
            "headings": extract_headings(body),
        }

        return email_record, None

    except Exception as error:
        return None, f"Unhandled parse error: {error}"

# **8. Test the Parser on a Small Sample**

Before processing thousands of emails, this section tests the parser on a small sample. This helps confirm that the extraction logic is working and that mandatory field validation is correctly filtering out problematic files.

In [13]:
sample_results = []

for file_path in email_files[:10]:
    parsed_email, error = parse_email_file(file_path, SUBSET_MAILDIR)

    sample_results.append({
        "file": str(file_path.relative_to(SUBSET_MAILDIR)),
        "parsed": parsed_email is not None,
        "error": error,
        "subject": parsed_email["subject"] if parsed_email else None
    })

sample_results

[{'file': 'kaminski-v/_sent_mail/1.',
  'parsed': True,
  'error': None,
  'subject': 'A resume for Londom'},
 {'file': 'kaminski-v/_sent_mail/10.',
  'parsed': False,
  'error': 'Missing mandatory field: body',
  'subject': None},
 {'file': 'kaminski-v/_sent_mail/100.',
  'parsed': False,
  'error': 'Missing mandatory field: subject',
  'subject': None},
 {'file': 'kaminski-v/_sent_mail/1000.',
  'parsed': True,
  'error': None,
  'subject': 'update on energy book'},
 {'file': 'kaminski-v/_sent_mail/1001.',
  'parsed': False,
  'error': 'Missing mandatory field: body',
  'subject': None},
 {'file': 'kaminski-v/_sent_mail/1002.',
  'parsed': True,
  'error': None,
  'subject': 'To_do'},
 {'file': 'kaminski-v/_sent_mail/1003.',
  'parsed': False,
  'error': 'Missing mandatory field: body',
  'subject': None},
 {'file': 'kaminski-v/_sent_mail/1004.',
  'parsed': True,
  'error': None,
  'subject': 'Re: Real Option Conference'},
 {'file': 'kaminski-v/_sent_mail/1005.',
  'parsed': True,
 

# **9. Design the SQLite Database Schema**

The database schema is normalized into two main tables:

1. emails stores one row per parsed email.
2. email_recipients stores To, CC, and BCC recipients in a separate table.

This avoids storing recipient lists as comma-separated strings and allows efficient querying by recipient type.

The schema also includes duplicate tracking columns required by the assignment:

* is_duplicate
* duplicate_of
* notification_sent
* notification_date

Indexes are created on date, from_address, and subject to support common analytical queries.

In [14]:
schema_sql = """
DROP TABLE IF EXISTS email_recipients;
DROP TABLE IF EXISTS parse_failures;
DROP TABLE IF EXISTS pipeline_runs;
DROP TABLE IF EXISTS emails;

CREATE TABLE pipeline_runs (
    run_id INTEGER PRIMARY KEY AUTOINCREMENT,
    started_at TEXT NOT NULL,
    finished_at TEXT,
    status TEXT,
    total_files_found INTEGER DEFAULT 0,
    successfully_parsed INTEGER DEFAULT 0,
    failed INTEGER DEFAULT 0,
    duplicate_message_ids_skipped INTEGER DEFAULT 0,
    duplicates_flagged INTEGER DEFAULT 0,
    notes TEXT
);

CREATE TABLE emails (
    message_id TEXT PRIMARY KEY,
    date TEXT NOT NULL,
    from_address TEXT NOT NULL,
    subject TEXT NOT NULL,
    subject_normalized TEXT NOT NULL,
    body TEXT NOT NULL,
    body_normalized TEXT NOT NULL,
    body_hash TEXT NOT NULL,
    source_file TEXT NOT NULL,

    x_from TEXT,
    x_to TEXT,
    x_cc TEXT,
    x_bcc TEXT,
    x_folder TEXT,
    x_origin TEXT,
    content_type TEXT,
    has_attachment INTEGER DEFAULT 0,
    forwarded_content TEXT,
    quoted_content TEXT,
    headings TEXT,

    is_duplicate INTEGER DEFAULT 0,
    duplicate_of TEXT,
    similarity_score REAL,
    notification_sent INTEGER DEFAULT 0,
    notification_date TEXT,

    created_at TEXT DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (duplicate_of) REFERENCES emails(message_id)
);

CREATE TABLE email_recipients (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    message_id TEXT NOT NULL,
    recipient_address TEXT NOT NULL,
    recipient_type TEXT NOT NULL CHECK (recipient_type IN ('to', 'cc', 'bcc')),

    FOREIGN KEY (message_id) REFERENCES emails(message_id)
);

CREATE TABLE parse_failures (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_file TEXT NOT NULL,
    failure_reason TEXT NOT NULL,
    failed_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX idx_emails_date ON emails(date);
CREATE INDEX idx_emails_from_address ON emails(from_address);
CREATE INDEX idx_emails_subject ON emails(subject);
CREATE INDEX idx_emails_subject_normalized ON emails(subject_normalized);
CREATE INDEX idx_emails_body_hash ON emails(body_hash);
CREATE INDEX idx_emails_duplicate_status ON emails(is_duplicate);
CREATE INDEX idx_recipients_address ON email_recipients(recipient_address);
CREATE INDEX idx_recipients_type ON email_recipients(recipient_type);
"""

In [15]:
def initialize_database(db_path: Path, schema: str) -> None:
    if db_path.exists():
        db_path.unlink()

    with sqlite3.connect(db_path) as connection:
        connection.executescript(schema)

    SCHEMA_PATH.write_text(schema.strip(), encoding="utf-8")

    logger.info("Database initialized at %s", db_path)
    logger.info("Schema exported to %s", SCHEMA_PATH)

In [16]:
initialize_database(DB_PATH, schema_sql)

In [17]:
def start_pipeline_run(db_path: Path) -> int:
    started_at = datetime.now(timezone.utc).isoformat()

    with sqlite3.connect(db_path) as connection:
        cursor = connection.execute(
            """
            INSERT INTO pipeline_runs (
                started_at,
                status
            )
            VALUES (?, ?)
            """,
            (started_at, "running")
        )

        return cursor.lastrowid


def finish_pipeline_run(
    db_path: Path,
    run_id: int,
    stats: Dict,
    status: str = "success",
    notes: str = ""
) -> None:
    finished_at = datetime.now(timezone.utc).isoformat()

    with sqlite3.connect(db_path) as connection:
        connection.execute(
            """
            UPDATE pipeline_runs
            SET
                finished_at = ?,
                status = ?,
                total_files_found = ?,
                successfully_parsed = ?,
                failed = ?,
                duplicate_message_ids_skipped = ?,
                duplicates_flagged = ?,
                notes = ?
            WHERE run_id = ?
            """,
            (
                finished_at,
                status,
                stats.get("total_files_found", 0),
                stats.get("successfully_parsed", 0),
                stats.get("failed", 0),
                stats.get("duplicate_message_ids_skipped", 0),
                stats.get("duplicates_flagged", 0),
                notes,
                run_id
            )
        )

# **10. Insert Parsed Emails into the Database**

This section defines helper functions for inserting parsed email records into SQLite.

The emails table stores the main message-level fields, while the email_recipients table stores To, CC, and BCC addresses separately. The message_id primary key prevents exact duplicate inserts.

In [18]:
def insert_email_record(connection: sqlite3.Connection, record: Dict) -> bool:
    try:
        connection.execute(
            """
            INSERT INTO emails (
                message_id,
                date,
                from_address,
                subject,
                subject_normalized,
                body,
                body_normalized,
                body_hash,
                source_file,
                x_from,
                x_to,
                x_cc,
                x_bcc,
                x_folder,
                x_origin,
                content_type,
                has_attachment,
                forwarded_content,
                quoted_content,
                headings
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                record["message_id"],
                record["date"],
                record["from_address"],
                record["subject"],
                record["subject_normalized"],
                record["body"],
                record["body_normalized"],
                record["body_hash"],
                record["source_file"],
                record["x_from"],
                record["x_to"],
                record["x_cc"],
                record["x_bcc"],
                record["x_folder"],
                record["x_origin"],
                record["content_type"],
                int(record["has_attachment"]),
                record["forwarded_content"],
                record["quoted_content"],
                record["headings"],
            )
        )

        recipient_rows = []

        for address in record["to_addresses"]:
            recipient_rows.append((record["message_id"], address, "to"))

        for address in record["cc_addresses"]:
            recipient_rows.append((record["message_id"], address, "cc"))

        for address in record["bcc_addresses"]:
            recipient_rows.append((record["message_id"], address, "bcc"))

        connection.executemany(
            """
            INSERT INTO email_recipients (
                message_id,
                recipient_address,
                recipient_type
            )
            VALUES (?, ?, ?)
            """,
            recipient_rows
        )

        return True

    except sqlite3.IntegrityError:
        return False

def log_parse_failure_to_db(
    connection: sqlite3.Connection,
    source_file: str,
    failure_reason: str
) -> None:
    connection.execute(
        """
        INSERT INTO parse_failures (
            source_file,
            failure_reason
        )
        VALUES (?, ?)
        """,
        (source_file, failure_reason)
    )

# **11. Run the Full Extraction Pipeline**

This section processes all discovered email files in the selected mailbox subset.

For each file, the pipeline:

1. Attempts to parse the email.
2. Validates all mandatory fields.
3. Inserts valid records into SQLite.
4. Logs failed parses with the file path and reason.
5. Tracks field-level completeness for reporting.

In [19]:
def run_extraction_pipeline(email_files: List[Path], root_path: Path, db_path: Path) -> Dict:
    stats = {
        "total_files_found": len(email_files),
        "successfully_parsed": 0,
        "failed": 0,
        "duplicate_message_ids_skipped": 0,
        "missing_message_id_count": 0,
        "missing_or_invalid_date_count": 0,
        "missing_from_count": 0,
        "missing_to_count": 0,
        "missing_subject_count": 0,
        "missing_body_count": 0,
        "malformed_email_count": 0,
        "field_completeness": {},
        "selected_mailboxes": CONFIG["selected_mailboxes"],
    }

    fields_to_track = [
        "message_id",
        "date",
        "from_address",
        "subject",
        "body",
        "body_hash",
        "cc_addresses",
        "bcc_addresses",
        "x_from",
        "x_to",
        "x_folder",
        "x_origin",
        "content_type",
        "has_attachment",
        "forwarded_content",
        "quoted_content",
        "headings",
    ]

    field_counts = {field: 0 for field in fields_to_track}

    with open(ERROR_LOG_PATH, "w", encoding="utf-8") as error_log:
        with sqlite3.connect(db_path) as connection:
            for index, file_path in enumerate(email_files, start=1):
                record, error = parse_email_file(file_path, root_path)

                if record is None:
                    stats["failed"] += 1

                    source_file = str(file_path.relative_to(root_path))
                    failure_reason = error or "Unknown parse error"

                    error_log.write(f"{source_file} | {failure_reason}\n")

                    log_parse_failure_to_db(
                        connection,
                        source_file,
                        failure_reason
                    )

                    error_lower = failure_reason.lower()

                    if "message_id" in error_lower:
                        stats["missing_message_id_count"] += 1
                    elif "date" in error_lower:
                        stats["missing_or_invalid_date_count"] += 1
                    elif "from_address" in error_lower:
                        stats["missing_from_count"] += 1
                    elif "to_addresses" in error_lower:
                        stats["missing_to_count"] += 1
                    elif "subject" in error_lower:
                        stats["missing_subject_count"] += 1
                    elif "body" in error_lower:
                        stats["missing_body_count"] += 1
                    else:
                        stats["malformed_email_count"] += 1

                    continue

                inserted = insert_email_record(connection, record)

                if not inserted:
                    stats["duplicate_message_ids_skipped"] += 1

                    source_file = str(file_path.relative_to(root_path))
                    error_log.write(
                        f"{source_file} | Duplicate message_id skipped: {record['message_id']}\n"
                    )

                    log_parse_failure_to_db(
                        connection,
                        source_file,
                        f"Duplicate message_id skipped: {record['message_id']}"
                    )

                    continue

                stats["successfully_parsed"] += 1

                for field in fields_to_track:
                    value = record.get(field)

                    if isinstance(value, list):
                        if len(value) > 0:
                            field_counts[field] += 1
                    elif value not in [None, "", False]:
                        field_counts[field] += 1

                if index % 1000 == 0:
                    logger.info("Processed %s / %s files", index, len(email_files))

    parsed_count = max(stats["successfully_parsed"], 1)

    stats["field_completeness"] = {
        field: {
            "count": count,
            "percentage": round((count / parsed_count) * 100, 2)
        }
        for field, count in field_counts.items()
    }

    EXTRACTION_STATS_PATH.write_text(
        json.dumps(stats, indent=2),
        encoding="utf-8"
    )

    return stats

In [20]:
run_id = start_pipeline_run(DB_PATH)

In [21]:
extraction_stats = run_extraction_pipeline(email_files, SUBSET_MAILDIR, DB_PATH)

extraction_stats

{'total_files_found': 75797,
 'successfully_parsed': 63949,
 'failed': 11848,
 'duplicate_message_ids_skipped': 0,
 'missing_message_id_count': 0,
 'missing_or_invalid_date_count': 0,
 'missing_from_count': 2,
 'missing_to_count': 2007,
 'missing_subject_count': 2032,
 'missing_body_count': 7807,
 'malformed_email_count': 0,
 'field_completeness': {'message_id': {'count': 63949, 'percentage': 100.0},
  'date': {'count': 63949, 'percentage': 100.0},
  'from_address': {'count': 63949, 'percentage': 100.0},
  'subject': {'count': 63949, 'percentage': 100.0},
  'body': {'count': 63949, 'percentage': 100.0},
  'body_hash': {'count': 63949, 'percentage': 100.0},
  'cc_addresses': {'count': 23078, 'percentage': 36.09},
  'bcc_addresses': {'count': 23078, 'percentage': 36.09},
  'x_from': {'count': 63949, 'percentage': 100.0},
  'x_to': {'count': 63949, 'percentage': 100.0},
  'x_folder': {'count': 63949, 'percentage': 100.0},
  'x_origin': {'count': 63949, 'percentage': 100.0},
  'content_typ

In [22]:
with sqlite3.connect(DB_PATH) as connection:
    parse_failure_count = connection.execute(
        "SELECT COUNT(*) FROM parse_failures"
    ).fetchone()[0]

    pipeline_run_count = connection.execute(
        "SELECT COUNT(*) FROM pipeline_runs"
    ).fetchone()[0]

print("Parse failures stored in DB:", parse_failure_count)
print("Pipeline run records:", pipeline_run_count)

Parse failures stored in DB: 11848
Pipeline run records: 1


# **12. Validate Database Contents**

After ingestion, this section checks the number of stored emails and recipient records. This confirms that the database contains structured email records and normalized recipient entries.

In [23]:
with sqlite3.connect(DB_PATH) as connection:
    email_count = connection.execute("SELECT COUNT(*) FROM emails").fetchone()[0]
    recipient_count = connection.execute("SELECT COUNT(*) FROM email_recipients").fetchone()[0]

print("Emails stored:", email_count)
print("Recipient records stored:", recipient_count)
print("Parse failures logged at:", ERROR_LOG_PATH)
print("Extraction statistics saved at:", EXTRACTION_STATS_PATH)

Emails stored: 63949
Recipient records stored: 368536
Parse failures logged at: /content/enron_email_pipeline/output/error_log.txt
Extraction statistics saved at: /content/enron_email_pipeline/output/extraction_stats.json


# **13. Sample Database Queries**

The assignment requires at least three sample queries that demonstrate the database works correctly.

The following queries show:

Email counts by sender
Emails within a date range
Emails that contain CC recipients

The queries are also exported to sample_queries.sql.

In [24]:
sample_queries_sql = """
-- Query 1: Count emails per sender
SELECT
    from_address,
    COUNT(*) AS email_count
FROM emails
GROUP BY from_address
ORDER BY email_count DESC
LIMIT 10;

-- Query 2: Find emails sent within a date range
SELECT
    message_id,
    date,
    from_address,
    subject
FROM emails
WHERE date BETWEEN '2001-01-01' AND '2001-12-31'
ORDER BY date
LIMIT 20;

-- Query 3: Find emails with CC recipients
SELECT
    e.message_id,
    e.date,
    e.from_address,
    e.subject,
    r.recipient_address AS cc_address
FROM emails e
JOIN email_recipients r
    ON e.message_id = r.message_id
WHERE r.recipient_type = 'cc'
LIMIT 20;
"""

SAMPLE_QUERIES_PATH.write_text(sample_queries_sql.strip(), encoding="utf-8")

print("Sample queries exported to:", SAMPLE_QUERIES_PATH)

Sample queries exported to: /content/enron_email_pipeline/output/sample_queries.sql


In [25]:
with sqlite3.connect(DB_PATH) as connection:
    print("Top senders:")
    top_senders = connection.execute("""
        SELECT from_address, COUNT(*) AS email_count
        FROM emails
        GROUP BY from_address
        ORDER BY email_count DESC
        LIMIT 10
    """).fetchall()

top_senders

Top senders:


[('kay.mann@enron.com', 14634),
 ('vince.kaminski@enron.com', 9373),
 ('mark.taylor@enron.com', 3168),
 ('shirley.crenshaw@enron.com', 921),
 ('rosalee.fleming@enron.com', 854),
 ('j.kaminski@enron.com', 772),
 ('sherri.sera@enron.com', 676),
 ('enron.announcements@enron.com', 559),
 ('ben.jacoby@enron.com', 491),
 ('vkaminski@aol.com', 477)]

In [26]:
with sqlite3.connect(DB_PATH) as connection:
    print("Emails from 2001:")
    emails_2001 = connection.execute("""
        SELECT message_id, date, from_address, subject
        FROM emails
        WHERE date BETWEEN '2001-01-01' AND '2001-12-31'
        ORDER BY date
        LIMIT 10
    """).fetchall()

emails_2001

Emails from 2001:


[('<2310129.1075856240385.JavaMail.evans@thyme>',
  '2001-01-01T14:23:00+00:00',
  'ps5@andrew.cmu.edu',
  'Merry Christmas'),
 ('<12771065.1075856384720.JavaMail.evans@thyme>',
  '2001-01-01T14:23:00+00:00',
  'ps5@andrew.cmu.edu',
  'Merry Christmas'),
 ('<18830741.1075856634553.JavaMail.evans@thyme>',
  '2001-01-01T14:23:00+00:00',
  'ps5@andrew.cmu.edu',
  'Merry Christmas'),
 ('<5302723.1075856240363.JavaMail.evans@thyme>',
  '2001-01-01T18:01:00+00:00',
  'vkaminski@aol.com',
  'A few comments'),
 ('<29143270.1075856384980.JavaMail.evans@thyme>',
  '2001-01-01T18:01:00+00:00',
  'vkaminski@aol.com',
  'A few comments'),
 ('<19638259.1075856634532.JavaMail.evans@thyme>',
  '2001-01-01T18:01:00+00:00',
  'vkaminski@aol.com',
  'A few comments'),
 ('<10466014.1075856529373.JavaMail.evans@thyme>',
  '2001-01-02T07:52:00+00:00',
  'vince.kaminski@enron.com',
  'book order'),
 ('<3371004.1075856240320.JavaMail.evans@thyme>',
  '2001-01-02T07:52:00+00:00',
  'vince.kaminski@enron.com',


In [27]:
with sqlite3.connect(DB_PATH) as connection:
    print("Emails with CC recipients:")
    cc_examples = connection.execute("""
        SELECT e.message_id, e.date, e.from_address, e.subject, r.recipient_address
        FROM emails e
        JOIN email_recipients r
            ON e.message_id = r.message_id
        WHERE r.recipient_type = 'cc'
        LIMIT 10
    """).fetchall()

cc_examples

Emails with CC recipients:


[('<18755653.1075856526751.JavaMail.evans@thyme>',
  '2001-01-12T14:53:00+00:00',
  'vince.kaminski@enron.com',
  'PLEASE NOTE THAT THE DATE FOR THE 1ST MEETING IS JANUARY 16',
  'vince.kaminski@enron.com'),
 ('<15937464.1075856526775.JavaMail.evans@thyme>',
  '2001-01-12T14:52:00+00:00',
  'vince.kaminski@enron.com',
  'asking for advice regarding Summer Associate position at Enron',
  'stinson.gibner@enron.com'),
 ('<15937464.1075856526775.JavaMail.evans@thyme>',
  '2001-01-12T14:52:00+00:00',
  'vince.kaminski@enron.com',
  'asking for advice regarding Summer Associate position at Enron',
  'vince.kaminski@enron.com'),
 ('<15937464.1075856526775.JavaMail.evans@thyme>',
  '2001-01-12T14:52:00+00:00',
  'vince.kaminski@enron.com',
  'asking for advice regarding Summer Associate position at Enron',
  'zimin.lu@enron.com'),
 ('<25981006.1075856500678.JavaMail.evans@thyme>',
  '2001-04-30T14:28:00+00:00',
  'vince.kaminski@enron.com',
  'Follow-up on SIAM Workshop',
  'vince.kaminski@enr

# **14. Duplicate Detection Strategy**

Duplicate detection is based on the assignment definition:

Two or more emails are considered duplicates if they share:

1. The same sender address
2. The same subject after removing prefixes such as Re: and Fwd:
3. Body content with at least 90 percent fuzzy similarity

Within each duplicate group, the earliest email is treated as the original record. Every later email in that group is marked as a duplicate and linked back to the original message through the duplicate_of column.

In [28]:
def fetch_duplicate_candidates(db_path: Path) -> Dict[Tuple[str, str], List[Dict]]:
    groups = {}

    with sqlite3.connect(db_path) as connection:
        connection.row_factory = sqlite3.Row

        rows = connection.execute("""
            SELECT
                message_id,
                date,
                from_address,
                subject,
                subject_normalized,
                body_normalized,
                body_hash
            FROM emails
            WHERE body_normalized IS NOT NULL
              AND body_normalized != ''
            ORDER BY from_address, subject_normalized, body_hash, date
        """).fetchall()

    for row in rows:
        record = dict(row)

        key = (
            record["from_address"],
            record["subject_normalized"]
        )

        if key not in groups:
            groups[key] = []

        groups[key].append(record)

    return {
        key: value
        for key, value in groups.items()
        if len(value) > 1
    }

In [29]:
def detect_duplicate_groups(
    candidate_groups: Dict[Tuple[str, str], List[Dict]],
    threshold: int
) -> List[Dict]:

    duplicate_groups = []

    for key, records in candidate_groups.items():
        records = sorted(records, key=lambda item: item["date"])

        hash_groups = {}

        for record in records:
            hash_groups.setdefault(record["body_hash"], []).append(record)

        used_message_ids = set()

        for body_hash, hash_records in hash_groups.items():
            if len(hash_records) > 1:
                original = hash_records[0]
                duplicates = []

                for duplicate in hash_records[1:]:
                    duplicates.append({
                        "record": duplicate,
                        "similarity_score": 100.0
                    })
                    used_message_ids.add(duplicate["message_id"])

                duplicate_groups.append({
                    "original": original,
                    "duplicates": duplicates
                })

        remaining_records = [
            record
            for record in records
            if record["message_id"] not in used_message_ids
        ]

        for i, original in enumerate(remaining_records):
            if original["message_id"] in used_message_ids:
                continue

            group_duplicates = []

            for candidate in remaining_records[i + 1:]:
                if candidate["message_id"] in used_message_ids:
                    continue

                similarity = fuzz.ratio(
                    original["body_normalized"],
                    candidate["body_normalized"]
                )

                if similarity >= threshold:
                    group_duplicates.append({
                        "record": candidate,
                        "similarity_score": similarity
                    })
                    used_message_ids.add(candidate["message_id"])

            if group_duplicates:
                duplicate_groups.append({
                    "original": original,
                    "duplicates": group_duplicates
                })

    return duplicate_groups

This version improves scalability by treating identical body hashes as instant duplicates and using fuzzy matching only for remaining records.

In [30]:
def flag_duplicates_in_database(db_path: Path, duplicate_groups: List[Dict]) -> Dict:
    total_flagged = 0
    group_sizes = []

    with sqlite3.connect(db_path) as connection:
        for group in duplicate_groups:
            original = group["original"]
            duplicates = group["duplicates"]

            group_sizes.append(1 + len(duplicates))

            for duplicate_entry in duplicates:
                duplicate = duplicate_entry["record"]
                similarity_score = duplicate_entry["similarity_score"]

                connection.execute(
                    """
                    UPDATE emails
                    SET
                        is_duplicate = 1,
                        duplicate_of = ?,
                        similarity_score = ?
                    WHERE message_id = ?
                    """,
                    (
                        original["message_id"],
                        similarity_score,
                        duplicate["message_id"]
                    )
                )

                total_flagged += 1

    average_group_size = round(sum(group_sizes) / len(group_sizes), 2) if group_sizes else 0

    return {
        "total_duplicate_groups_found": len(duplicate_groups),
        "total_emails_flagged": total_flagged,
        "average_group_size": average_group_size
    }

In [31]:
candidate_groups = fetch_duplicate_candidates(DB_PATH)

duplicate_groups = detect_duplicate_groups(
    candidate_groups,
    threshold=CONFIG["duplicate_similarity_threshold"]
)

duplicate_stats = flag_duplicates_in_database(DB_PATH, duplicate_groups)

duplicate_stats

{'total_duplicate_groups_found': 18334,
 'total_emails_flagged': 37123,
 'average_group_size': 3.02}

# **15. Generate Duplicate Report**

This section exports duplicates_report.csv, which lists every flagged duplicate email along with its original email, sender, subject, dates, and similarity score.

In [32]:
def generate_duplicates_report(db_path: Path, output_path: Path) -> int:
    with sqlite3.connect(db_path) as connection:
        connection.row_factory = sqlite3.Row

        rows = connection.execute("""
            SELECT
                duplicate.message_id AS duplicate_message_id,
                original.message_id AS original_message_id,
                duplicate.subject AS subject,
                duplicate.from_address AS from_address,
                duplicate.date AS duplicate_date,
                original.date AS original_date,
                duplicate.similarity_score AS similarity_score
            FROM emails duplicate
            JOIN emails original
                ON duplicate.duplicate_of = original.message_id
            WHERE duplicate.is_duplicate = 1
            ORDER BY duplicate.from_address, duplicate.subject, duplicate.date
        """).fetchall()

    with open(output_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        writer.writerow([
            "duplicate_message_id",
            "original_message_id",
            "subject",
            "from_address",
            "duplicate_date",
            "original_date",
            "similarity_score"
        ])

        for row in rows:
            writer.writerow([
                row["duplicate_message_id"],
                row["original_message_id"],
                row["subject"],
                row["from_address"],
                row["duplicate_date"],
                row["original_date"],
                row["similarity_score"]
            ])

    return len(rows)

In [33]:
duplicate_report_count = generate_duplicates_report(DB_PATH, DUPLICATES_REPORT_PATH)

print("Duplicate rows exported:", duplicate_report_count)
print("Duplicate report path:", DUPLICATES_REPORT_PATH)

Duplicate rows exported: 37123
Duplicate report path: /content/enron_email_pipeline/output/duplicates_report.csv


In [34]:
extraction_stats["duplicates_flagged"] = duplicate_report_count

finish_pipeline_run(
    DB_PATH,
    run_id,
    extraction_stats,
    status="success",
    notes="Notebook pipeline completed successfully"
)

In [35]:
with sqlite3.connect(DB_PATH) as connection:
    duplicate_db_count = connection.execute(
        "SELECT COUNT(*) FROM emails WHERE is_duplicate = 1"
    ).fetchone()[0]

print("Duplicates in database:", duplicate_db_count)
print("Rows in duplicate report:", duplicate_report_count)

Duplicates in database: 37123
Rows in duplicate report: 37123


# **16. Create Dry-Run Notification Email Drafts**

The assignment requires notification emails to be sent through Gmail MCP integration in live mode. For notebook safety, this section creates dry-run .eml notification drafts for flagged duplicate emails.

Each draft follows the required notification template and is saved in the output/replies/ folder. These files can be inspected before enabling live email sending.

In [36]:
def build_notification_email(row: sqlite3.Row) -> Tuple[str, str, str]:
    recipient = row["from_address"]
    subject = f"[Duplicate Notice] Re: {row['subject']}"

    current_timestamp = datetime.now(timezone.utc).isoformat()

    body = f"""This is an automated notification from the Email Deduplication System.

Your email has been identified as a potential duplicate:

  Your Email (Flagged):
    Message-ID:  {row['duplicate_message_id']}
    Date Sent:   {row['duplicate_date']}
    Subject:     {row['subject']}

  Original Email on Record:
    Message-ID:  {row['original_message_id']}
    Date Sent:   {row['original_date']}

  Similarity Score: {round(row['similarity_score'], 2)}%

If this was NOT a duplicate and you intended to send this email,
please reply with CONFIRM to restore it to active status.

No action is required if this is indeed a duplicate.
"""

    email_text = f"""To: {recipient}
Subject: {subject}
Date: {current_timestamp}
References: {row['duplicate_message_id']}
Content-Type: text/plain; charset=UTF-8

{body}
"""

    return recipient, subject, email_text

In [37]:
def create_notification_drafts(db_path: Path, replies_dir: Path) -> List[Dict]:
    replies_dir.mkdir(parents=True, exist_ok=True)

    with sqlite3.connect(db_path) as connection:
        connection.row_factory = sqlite3.Row

        rows = connection.execute("""
            SELECT
                duplicate.message_id AS duplicate_message_id,
                original.message_id AS original_message_id,
                duplicate.subject AS subject,
                duplicate.from_address AS from_address,
                duplicate.date AS duplicate_date,
                original.date AS original_date,
                duplicate.similarity_score AS similarity_score
            FROM emails duplicate
            JOIN emails original
                ON duplicate.duplicate_of = original.message_id
            WHERE duplicate.is_duplicate = 1
            ORDER BY duplicate.date DESC
        """).fetchall()

    draft_logs = []

    for row in rows:
        recipient, subject, email_text = build_notification_email(row)

        safe_id = hashlib.md5(row["duplicate_message_id"].encode("utf-8")).hexdigest()
        draft_path = replies_dir / f"duplicate_notice_{safe_id}.eml"

        draft_path.write_text(email_text, encoding="utf-8")

        draft_logs.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "recipient": recipient,
            "subject": subject,
            "status": "draft_created",
            "error": "",
            "draft_path": str(draft_path)
        })

    return draft_logs

In [38]:
draft_logs = create_notification_drafts(DB_PATH, REPLIES_DIR)

print("Notification drafts created:", len(draft_logs))
print("Draft output folder:", REPLIES_DIR)

Notification drafts created: 37123
Draft output folder: /content/enron_email_pipeline/output/replies


# **17. Generate Send Log**

In dry-run mode, no live emails are sent. Instead, the notebook writes a send_log.csv file showing which notification drafts were created.

For the final project repository, this same structure can be reused for live MCP sending by changing the status from draft_created to sent after the Gmail MCP tool successfully sends the email.

In [39]:
def write_send_log(logs: List[Dict], output_path: Path) -> None:
    with open(output_path, "w", newline="", encoding="utf-8") as file:
        fieldnames = [
            "timestamp",
            "recipient",
            "subject",
            "status",
            "error",
            "draft_path"
        ]

        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(logs)

In [40]:
write_send_log(draft_logs, SEND_LOG_PATH)

print("Send log written to:", SEND_LOG_PATH)

Send log written to: /content/enron_email_pipeline/output/send_log.csv


# **18. MCP Live Email Integration Notes**

This notebook uses dry-run .eml draft generation for safety. In the complete repository submission, live sending should be handled through a Gmail-compatible MCP server.

The intended live-mode flow is:

1. Detect duplicate emails.
2. Select the latest duplicate in each duplicate group.
3. Build the notification email using the required template.
4. Send the notification through the Gmail MCP send_email tool.
5. Update the database columns:
*  notification_sent = 1
* notification_date = current timestamp
6. Append the result to output/send_log.csv.

The repository should include a redacted mcp_config.json.example file with placeholder credentials and an AI_USAGE.md section explaining the MCP setup, prompts used, and any setup issues.

In [41]:
mcp_config_example = {
    "mcpServers": {
        "gmail": {
            "command": "npx",
            "args": [
                "-y",
                "gmail-mcp-server"
            ],
            "env": {
                "GOOGLE_CLIENT_ID": "YOUR_CLIENT_ID",
                "GOOGLE_CLIENT_SECRET": "YOUR_CLIENT_SECRET",
                "GOOGLE_REFRESH_TOKEN": "YOUR_REFRESH_TOKEN"
            }
        }
    }
}

mcp_config_path = OUTPUT_DIR / "mcp_config.json.example"

mcp_config_path.write_text(
    json.dumps(mcp_config_example, indent=2),
    encoding="utf-8"
)

print("MCP config example written to:", mcp_config_path)

MCP config example written to: /content/enron_email_pipeline/output/mcp_config.json.example


## **18A. Live Gmail MCP Sending Design**

The notebook already generates dry-run .eml notification drafts for every flagged duplicate. For live submission mode, the same notification payload can be sent through a Gmail-compatible MCP server.

Live sending is intentionally separated from dry-run mode because duplicate detection can produce thousands of flagged emails. Sending thousands of live emails during testing would be unsafe. Therefore, this section implements a controlled live-mode function that only sends emails when SEND_LIVE = True.

The live-mode workflow is:

1. Load Gmail MCP configuration from mcp_config.json
2. Read flagged duplicate records from SQLite
3. Build the notification email using the assignment template
4. Send the email through the MCP Gmail send_email tool
5. Update notification_sent and notification_date
6. Log each send attempt in send_log.csv

For safety, this notebook leaves SEND_LIVE = False by default.

In [42]:
SEND_LIVE = False
MAX_LIVE_SENDS = 3

TEST_RECIPIENT_EMAIL = "enron.pipeline.test@gmail.com"
USE_TEST_RECIPIENT_ONLY = True

MCP_CONFIG_PATH = OUTPUT_DIR / "mcp_config.json"

print("Live sending enabled:", SEND_LIVE)
print("Maximum live emails allowed in this run:", MAX_LIVE_SENDS)
print("Live test recipient:", TEST_RECIPIENT_EMAIL)

Live sending enabled: False
Maximum live emails allowed in this run: 3
Live test recipient: enron.pipeline.test@gmail.com


# **18B. Load MCP Gmail Configuration**

The live Gmail sender expects a local mcp_config.json file. This file should contain the Gmail MCP server configuration and OAuth credentials.

Credentials should never be committed to GitHub. Only mcp_config.json.example should be included in the repository.

In [43]:
def load_mcp_config(config_path: Path) -> Dict:
    if not config_path.exists():
        raise FileNotFoundError(
            f"MCP config file not found at {config_path}. "
            "Create mcp_config.json locally using mcp_config.json.example as a template."
        )

    with open(config_path, "r", encoding="utf-8") as file:
        config = json.load(file)

    if "mcpServers" not in config or "gmail" not in config["mcpServers"]:
        raise ValueError("Invalid MCP config. Expected mcpServers.gmail section.")

    return config

# **18C. Fetch Pending Duplicate Notifications**

This function selects duplicate emails that have not yet been notified. The query joins each duplicate email back to its original message so that the notification email can include both records.

In [44]:
def fetch_pending_duplicate_notifications(db_path: Path, limit: Optional[int] = None) -> List[sqlite3.Row]:
    query = """
        SELECT
            duplicate.message_id AS duplicate_message_id,
            original.message_id AS original_message_id,
            duplicate.subject AS subject,
            duplicate.from_address AS from_address,
            duplicate.date AS duplicate_date,
            original.date AS original_date,
            duplicate.similarity_score AS similarity_score
        FROM emails duplicate
        JOIN emails original
            ON duplicate.duplicate_of = original.message_id
        WHERE duplicate.is_duplicate = 1
          AND duplicate.notification_sent = 0
        ORDER BY duplicate.date DESC
    """

    if limit is not None:
        query += " LIMIT ?"

    with sqlite3.connect(db_path) as connection:
        connection.row_factory = sqlite3.Row

        if limit is not None:
            rows = connection.execute(query, (limit,)).fetchall()
        else:
            rows = connection.execute(query).fetchall()

    return rows

# **18D. MCP Send Email Wrapper**

In Claude Code, the Gmail MCP server exposes a send_email tool. The exact function call depends on the MCP server selected, but the payload should contain the recipient, subject, body, and references header.

The function below is written as a clear integration wrapper. In dry-run mode, it records what would be sent. In live mode, this wrapper is where the MCP Gmail send_email call is made.

In [45]:
def send_email_via_gmail_mcp(
    recipient: str,
    subject: str,
    body: str,
    references: str,
    send_live: bool = False
) -> Dict:
    timestamp = datetime.now(timezone.utc).isoformat()
    original_recipient = recipient

    if USE_TEST_RECIPIENT_ONLY:
        recipient = TEST_RECIPIENT_EMAIL

    if not send_live:
        return {
            "timestamp": timestamp,
            "recipient": recipient,
            "original_recipient": original_recipient,
            "subject": subject,
            "status": "dry_run_not_sent",
            "error": "",
            "references": references
        }

    try:
        mcp_config = load_mcp_config(MCP_CONFIG_PATH)

        # In Claude Code, this step is executed by the Gmail MCP server's send_email tool.
        # The expected MCP tool payload is:
        #
        # send_email({
        #     "to": recipient,
        #     "subject": subject,
        #     "body": body,
        #     "headers": {
        #         "References": references
        #     }
        # })
        #
        # The notebook keeps this as an integration wrapper because direct MCP tool
        # execution depends on the local Claude Code MCP runtime.

        raise NotImplementedError(
            "Live MCP sending must be executed inside Claude Code or another MCP-enabled client "
            "with the Gmail MCP server configured."
        )

    except Exception as error:
        return {
            "timestamp": timestamp,
            "recipient": recipient,
            "subject": subject,
            "status": "failed",
            "error": str(error),
            "references": references
        }

# **18E. Update Notification Status in SQLite**

After a notification is sent successfully, the database should be updated so that the same duplicate is not notified again in future runs.

In [51]:
def mark_notification_sent(
    db_path: Path,
    duplicate_message_id: str,
    notification_date: str
) -> None:
    with sqlite3.connect(db_path) as connection:
        connection.execute(
            """
            UPDATE emails
            SET
                notification_sent = 1,
                notification_date = ?
            WHERE message_id = ?
            """,
            (notification_date, duplicate_message_id)
        )

For safe live validation, all live MCP emails are routed to a dedicated dummy Gmail account instead of historical Enron email addresses. The live run is capped at 3 messages to prove external service integration without causing accidental mass emailing.

# **18F. Controlled Live Notification Runner**

This runner connects the previous pieces. It fetches pending duplicate notifications, builds the required email body, and either performs a dry-run or sends through MCP live mode.

MAX_LIVE_SENDS is intentionally small to prevent accidental mass emailing.

In [47]:
def run_mcp_notification_sender(
    db_path: Path,
    send_log_path: Path,
    send_live: bool = False,
    max_live_sends: int = 1
) -> List[Dict]:

    limit = max_live_sends if send_live else 5
    pending_rows = fetch_pending_duplicate_notifications(db_path, limit=3)

    logs = []

    for row in pending_rows:
        recipient, subject, email_text = build_notification_email(row)

        result = send_email_via_gmail_mcp(
            recipient=recipient,
            subject=subject,
            body=email_text,
            references=row["duplicate_message_id"],
            send_live=send_live
        )

        logs.append(result)

        if result["status"] == "sent":
            mark_notification_sent(
                db_path=db_path,
                duplicate_message_id=row["duplicate_message_id"],
                notification_date=result["timestamp"]
            )

    existing_logs = []

    if send_log_path.exists():
        with open(send_log_path, "r", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            existing_logs = list(reader)

    combined_logs = existing_logs + logs

    with open(send_log_path, "w", newline="", encoding="utf-8") as file:
        fieldnames = [
            "timestamp",
            "recipient",
            "original_recipient",
            "subject",
            "status",
            "error",
            "references"

        ]

        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()

        for log in combined_logs:
            writer.writerow({
                "timestamp": log.get("timestamp", ""),
                "recipient": log.get("recipient", ""),
                "original_recipient": log.get("original_recipient", ""),
                "subject": log.get("subject", ""),
                "status": log.get("status", ""),
                "error": log.get("error", ""),
                "references": log.get("references", "")
            })

    return logs

In [48]:
mcp_send_logs = run_mcp_notification_sender(
    db_path=DB_PATH,
    send_log_path=SEND_LOG_PATH,
    send_live=SEND_LIVE,
    max_live_sends=MAX_LIVE_SENDS
)

mcp_send_logs

[{'timestamp': '2026-05-02T19:24:04.183739+00:00',
  'recipient': 'enron.pipeline.test@gmail.com',
  'original_recipient': 'bmccall@wcnet.net',
  'subject': '[Duplicate Notice] Re: Palacios calling',
  'status': 'dry_run_not_sent',
  'error': '',
  'references': '<4824154.1075845849556.JavaMail.evans@thyme>'},
 {'timestamp': '2026-05-02T19:24:04.183751+00:00',
  'recipient': 'enron.pipeline.test@gmail.com',
  'original_recipient': 'bmccall@wcnet.net',
  'subject': '[Duplicate Notice] Re: Palacios calling',
  'status': 'dry_run_not_sent',
  'error': '',
  'references': '<32191524.1075845911247.JavaMail.evans@thyme>'},
 {'timestamp': '2026-05-02T19:24:04.183760+00:00',
  'recipient': 'enron.pipeline.test@gmail.com',
  'original_recipient': 'smarra@isda.org',
  'subject': '[Duplicate Notice] Re: NEXT US REGULATORY CALL....',
  'status': 'dry_run_not_sent',
  'error': '',
  'references': '<755156.1075860953154.JavaMail.evans@thyme>'}]

# **18G. Live MCP Execution Note**

The notebook includes the live-mode control flow, database updates, and logging structure required for MCP sending. However, actual live email delivery must be executed from an MCP-enabled environment such as Claude Code with the Gmail MCP server configured.

For repository submission, the same logic should be placed in a script such as notification_sender.py, where the MCP Gmail send_email tool call is executed when the command is run with:

 Run the live pipeline from terminal:

python main.py --send-live --max-live-sends 1

# **19. Final Database Summary**

This section summarizes the final state of the database after extraction and duplicate detection.

It reports:

* Total stored emails
* Total recipient records
* Total flagged duplicates
* Total notification drafts
* Output file locations

In [49]:
with sqlite3.connect(DB_PATH) as connection:
    total_emails = connection.execute(
        "SELECT COUNT(*) FROM emails"
    ).fetchone()[0]

    total_recipients = connection.execute(
        "SELECT COUNT(*) FROM email_recipients"
    ).fetchone()[0]

    total_duplicates = connection.execute(
        "SELECT COUNT(*) FROM emails WHERE is_duplicate = 1"
    ).fetchone()[0]

    total_with_attachments = connection.execute(
        "SELECT COUNT(*) FROM emails WHERE has_attachment = 1"
    ).fetchone()[0]

summary = {
    "total_email_files_discovered": len(email_files),
    "total_emails_stored": total_emails,
    "total_recipient_records": total_recipients,
    "total_duplicates_flagged": total_duplicates,
    "total_notification_drafts": len(draft_logs),
    "total_emails_with_possible_attachments": total_with_attachments,
    "database_path": str(DB_PATH),
    "error_log_path": str(ERROR_LOG_PATH),
    "duplicates_report_path": str(DUPLICATES_REPORT_PATH),
    "send_log_path": str(SEND_LOG_PATH),
    "schema_path": str(SCHEMA_PATH),
    "sample_queries_path": str(SAMPLE_QUERIES_PATH),
    "mcp_config_example_path": str(mcp_config_path),
}

summary

{'total_email_files_discovered': 75797,
 'total_emails_stored': 63949,
 'total_recipient_records': 368536,
 'total_duplicates_flagged': 37123,
 'total_notification_drafts': 37123,
 'total_emails_with_possible_attachments': 10217,
 'database_path': '/content/enron_email_pipeline/database/enron_emails.db',
 'error_log_path': '/content/enron_email_pipeline/output/error_log.txt',
 'duplicates_report_path': '/content/enron_email_pipeline/output/duplicates_report.csv',
 'send_log_path': '/content/enron_email_pipeline/output/send_log.csv',
 'schema_path': '/content/enron_email_pipeline/output/schema.sql',
 'sample_queries_path': '/content/enron_email_pipeline/output/sample_queries.sql',
 'mcp_config_example_path': '/content/enron_email_pipeline/output/mcp_config.json.example'}

# **20. Conclusion**

This notebook built a complete email extraction and deduplication pipeline for the Enron Email Dataset.

The final pipeline:

* Recursively discovered raw email files from a selected subset of Enron mailboxes
* Parsed mandatory and optional email fields from RFC 2822 messages
* Logged malformed or incomplete records instead of allowing the pipeline to fail
* Stored valid records in a normalized SQLite database
* Enforced uniqueness through message_id
* Indexed important fields for efficient querying
* Detected duplicates using sender, normalized subject, and fuzzy body similarity
* Flagged duplicate records in the database
* Generated duplicates_report.csv
* Created dry-run notification .eml files
* Produced supporting output files for schema, sample queries, error logs, and send logs

The implementation is structured to support both notebook-based demonstration and later conversion into a command-line project using main.py, schema.sql, README.md, AI_USAGE.md, and MCP-based live Gmail notification sending.

In [50]:
TEST_RECIPIENT_EMAIL = "enron.pipeline.test@gmail.com"

pending_rows = fetch_pending_duplicate_notifications(DB_PATH, limit=3)

live_test_payloads = []

for row in pending_rows:
    original_recipient, subject, email_text = build_notification_email(row)

    live_test_payloads.append({
        "duplicate_message_id": row["duplicate_message_id"],
        "to": TEST_RECIPIENT_EMAIL,
        "original_recipient": original_recipient,
        "subject": subject,
        "body": email_text
    })

print(f"Prepared {len(live_test_payloads)} live test emails")
live_test_payloads

Prepared 3 live test emails


[{'duplicate_message_id': '<4824154.1075845849556.JavaMail.evans@thyme>',
  'to': 'enron.pipeline.test@gmail.com',
  'original_recipient': 'bmccall@wcnet.net',
  'subject': '[Duplicate Notice] Re: Palacios calling',
  'body': 'To: bmccall@wcnet.net\nSubject: [Duplicate Notice] Re: Palacios calling\nDate: 2026-05-02T19:24:04.408030+00:00\nReferences: <4824154.1075845849556.JavaMail.evans@thyme>\nContent-Type: text/plain; charset=UTF-8\n\nThis is an automated notification from the Email Deduplication System.\n\nYour email has been identified as a potential duplicate:\n\n  Your Email (Flagged):\n    Message-ID:  <4824154.1075845849556.JavaMail.evans@thyme>\n    Date Sent:   2002-05-09T14:23:00+00:00\n    Subject:     Palacios calling\n\n  Original Email on Record:\n    Message-ID:  <29887046.1075845712932.JavaMail.evans@thyme>\n    Date Sent:   2002-05-09T14:23:00+00:00\n\n  Similarity Score: 100.0%\n\nIf this was NOT a duplicate and you intended to send this email,\nplease reply with CON

In [53]:
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).isoformat()

sent_ids = [
    "<755156.1075860953154.JavaMail.evans@thyme>",
    "<32191524.1075845911247.JavaMail.evans@thyme>",
    "<4824154.1075845849556.JavaMail.evans@thyme>"
]

for msg_id in sent_ids:
    mark_notification_sent(
        DB_PATH,
        msg_id,
        timestamp
    )

print("Updated successfully for", len(sent_ids), "emails")

Updated successfully for 3 emails


In [54]:
with sqlite3.connect(DB_PATH) as conn:
    rows = conn.execute("""
        SELECT message_id, notification_sent
        FROM emails
        WHERE message_id IN (
            '<755156.1075860953154.JavaMail.evans@thyme>',
            '<32191524.1075845911247.JavaMail.evans@thyme>',
            '<4824154.1075845849556.JavaMail.evans@thyme>'
        )
    """).fetchall()

rows

[('<32191524.1075845911247.JavaMail.evans@thyme>', 1),
 ('<4824154.1075845849556.JavaMail.evans@thyme>', 1),
 ('<755156.1075860953154.JavaMail.evans@thyme>', 1)]